# Quick test of umap version of KNN

Jan 22, 2026

In [ ]:
import matplotlib.pyplot as plt
from rail.core.stage import RailStage
from rail.core.data import TableHandle, ModelHandle
from rail.estimation.algos.knn_umap import UmapKnnInformer, UmapKnnEstimator
from rail.estimation.algos.umap import UMAPTrainer 
from rail.utils.path_utils import find_rail_file
import numpy as np
import tables_io

In [ ]:
DS = RailStage.data_store
DS.__class__.allow_overwrite = True

In [ ]:
trainFile = find_rail_file('examples_data/testdata/test_dc2_training_9816.hdf5')
testFile = find_rail_file('examples_data/testdata/test_dc2_validation_9816.hdf5')
training_data = DS.read_file("training_data", TableHandle, trainFile)
test_data = DS.read_file("test_data", TableHandle, testFile)

In [ ]:
# set up data info

xbands = ['u','g','r','i','z','y']
ref_band = "mag_i_lsst"
nondetect_val=99.
mag_limits = {}
bands = []
for band in xbands:
    #mag_limits[f"mag_{band}_lsst"] = 29.0
    bands.append(f"mag_{band}_lsst")

In [ ]:
umap_dict = dict(
    # I/O
    hdf5_groupname='photometry',
    # feature construction (KNN-compatible)
    bands=bands,
    ref_band=ref_band,
    only_colors=True,
    nondetect_val=nondetect_val,
    #mag_limits=mag_limits,
    # UMAP hyperparameters
    n_neighbors=10,
    min_dist=0.05,
    metric="manhattan",
    n_components=3,
    init="spectral",
    verbose=True,
    n_epochs=100,
)

trainer = UMAPTrainer.make_stage(
    name="umap_trainer",
    model="UMAP_reducer_model.pkl",
    **umap_dict
)

In [ ]:
model_handle = trainer.inform(training_data)

In [ ]:
# run the KNN informer to create KD tree model

In [ ]:
umap_train = UmapKnnInformer.make_stage(name="inform_umap", model=trainer.get_handle("model"), 
                                        output_model="knn_umap_model.pkl",
                                        hdf5_groupname="photometry")

In [ ]:
umap_train.inform(training_data)

In [ ]:
# run the estimate stage to make PDFs

In [ ]:
umap_test = UmapKnnEstimator.make_stage(name="estimate_umap", model=umap_train.get_handle('output_model'), 
                                        output="knn_umap_output.hdf5",
                                        hdf5_groupname="photometry")

In [ ]:
results = umap_test.estimate(test_data)

In [ ]:
zmode = results().ancil['zmode'].flatten()

In [ ]:
sz = test_data()['photometry']['redshift']

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(sz, zmode, s=2, c='k')
plt.plot([0,3], [0,3], 'r--')
plt.xlabel("redshift", fontsize=14)
plt.ylabel("z mode", fontsize=14);

In [ ]:
whichgal = 1544
fig, axs = plt.subplots(1, 1, figsize=(8,5))
results().plot_native(key=whichgal,axes=axs, label=f"PDF for galaxy {whichgal}")
plt.axvline(sz[whichgal], c='r',ls='--', label="true redshift");
plt.xlabel("redshift")
plt.ylabel("PDF")
plt.legend(loc='upper right');